# Interconnecting an OVS Network with an LLM via FabNet 

This notebook demonstrates how to set up and connect an Open vSwitch (OVS) network across multiple FABRIC sites, enabling communication with an LLM running in a different slice over FabNet.  

### Background: OVS  

Open vSwitch ([OVS](https://www.openvswitch.org/)) is an open-source, multi-layer virtual switch designed for virtualized environments. It plays a key role in many software-defined networking (SDN) and virtualization platforms.  

### Target FABRIC Topology  

In this setup, three OVS bridges are deployed across three different FABRIC sites, forming a ring topology. Each bridge is connected to a VM at the same site, and both the bridge and the VM are linked to the FABNetv4 network.  

We then demonstrate traffic flow between the three VMs and run network tests—including `ping`, `iperf`, `traceroute`, and `mtr`. The test results are forwarded to an LLM running on a separate FABRIC slice connected to FabNetv4, and the LLM generates responses based on the analysis.  

A high-level view of the topology is illustrated below.  

<img src="../figure/openvswitch.png" width="70%"><br>  

### Host Placement Considerations  

Due to NVIDIA/Mellanox constraints, when using `NIC_Basic` for an OVS bridge experiment, it is recommended to deploy the bridge VM on a separate host from the VMs connected to the bridge.  

However, this restriction does not apply to `NIC_ConnectX_5` and `NIC_ConnectX_6` configurations.

## Step 1: Import the FABlib Library

In [ ]:
from ipaddress import ip_address, IPv4Address, IPv4Network
import ipaddress
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager


fablib = fablib_manager()
fablib.show_config();

## Step 2: Check your existing slices


In [ ]:
try:
    for slice in fablib.get_slices():
        print(f"{slice}")
except Exception as e:
    print(f"Exception: {e}")

## Step 3: Observe the Slice's Attributes

### Select the slice 

In [ ]:
slice_name= f"slice-generic-cluster"


In [ ]:
try:
    slice = fablib.get_slice(name=slice_name)
    print(f"{slice}")
except Exception as e:
    print(f"Exception: {e}")

### Print the Node List

In [ ]:
try:
    slice = fablib.get_slice(name=slice_name)

    print(f"{slice.list_nodes()}")
except Exception as e:
    print(f"Exception: {e}")

### Print the Node Details (Optional)

In [ ]:
#try:
#    slice = fablib.get_slice(name=slice_name)
#    for node in slice.get_nodes():
#        print(f"{node}")
#except Exception as e:
#    print(f"Exception: {e}")

### Print the Interfaces (Optional)

In [ ]:
#try:
#    slice = fablib.get_slice(name=slice_name)
#    print(f"{slice.list_interfaces()}")
#except Exception as e:
#    print(f"Exception: {e}")

### Create a new bridge, enable the spanning tree protocol on necessary interfaces

In [ ]:
try:
    for node in slice.get_nodes():
        if node.get_name().endswith("-br"):
            stdout, stderr = node.execute('sudo ovs-vsctl add-br br0')
            for interface in node.get_interfaces():
                stdout, stderr = node.execute(f'sudo ovs-vsctl add-port br0 {interface.get_physical_os_interface_name()}')
                #Remove IP addresses for all interfaces
                stdout, stderr = node.execute(f'sudo ifconfig {interface.get_physical_os_interface_name()} 0')
    
            #bring the bridge up
            stdout, stderr = node.execute('sudo ifconfig br0 up')
    print("Done")
except Exception as e:
    print(f"Exception: {e}")

### Enable Spanning tree and confirm

In [ ]:
for node in slice.get_nodes():
    if node.get_name().endswith("-br"):
        stdout, stderr = node.execute('sudo ovs-vsctl set bridge br0 stp_enable=true')

In [ ]:
for node in slice.get_nodes():
    if node.get_name().endswith("-br"):
        stdout, stderr = node.execute('sudo ovs-appctl stp/show')
        print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
        print()

## Verify connectivity between host nodes

In [ ]:
#host1 = slice.get_node(name=f'{site1.lower()}')
#host2 = slice.get_node(name=f'{site2.lower()}')
#host3 = slice.get_node(name=f'{site3.lower()}')
node_name_1 = "s1-vm"
node_name_2 = "s2-vm"
node_name_3 = "s3-vm"

host1 = slice.get_node(name=node_name_1)
host2 = slice.get_node(name=node_name_2)
host3 = slice.get_node(name=node_name_3)

In [ ]:
# Ping test
host2_ip_addr = host2.get_interface(network_name=f"s2_layer2").get_ip_addr()
host3_ip_addr = host3.get_interface(network_name=f"s3_layer2").get_ip_addr()
stdout, stderr = host1.execute(f'ping {host2_ip_addr} -c 5')

# Ping test
stdout, stderr = host1.execute(f'ping {host3_ip_addr} -c 5')

## Enable Access in all nodes to access other Notes Across FABRIC Internet(FabNet)

Configure all the nodes in the slice connected to FabNetv4 to be accessible from any VM running across FABRIC on FabNetV4 by setting up the necessary routes.


In [ ]:
slice = fablib.get_slice(slice_name)
for n in slice.get_nodes():
    network_name = f"{n.get_site().lower()}_l3"
    fabnet_network = slice.get_network(network_name)

    n.add_route(subnet=fablib.FABNETV4_SUBNET, next_hop=fabnet_network.get_gateway())
    n.config_routes()

    stdout, stderr = n.execute("sudo ip route list")
    print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
    print()

## Ollama Node Configuration in a Separate Slice

Please update the **IP address, model in use, and port number** for the Ollama node operating in a different slice and connected to **FabNetV4**. This LLM will be utilized to analyze test results from hosts connected to the **OVS Bridges** in the current slice.

In [ ]:
ollama_fabnet_ip_addr = "10.133.134.2"
ollama_model = "deepseek-r1:7b"
ollama_port = "11434"

## Execute Tests

We will now conduct a series of tests between `host1`, `host2`, and `host3`, which are interconnected through a ring of **OVS switches**. The test results will be sent to the **LLM**, running in a different slice, for analysis.

### Upload test scripts

In [ ]:
host1.upload_directory("./tools", ".")

### Execute Ping Tests

Perform **ping tests** from `host1` to both `host2` and `host3`. Forward the output to the **LLM** for comparative analysis.

In [ ]:
print(f"Running ping test SRC: {host1.get_name()} DEST: {host2.get_name()}, {host3.get_name()}")

stdout, stderr = host1.execute(f'python3 tools/net_llm_tester.py --test_type ping --dest_ips {host2_ip_addr} {host3_ip_addr} --ollama_model {ollama_model} --ollama_host {ollama_fabnet_ip_addr}  --ollama_port {ollama_port}')

### Execute MTR Tests  

Run **MTR (My Traceroute) tests** from `host1` to both `host2` and `host3`. Send the output to the **LLM** for comparative analysis.

In [ ]:
print(f"Running mtr test SRC: {host1.get_name()} DEST: {host2.get_name()}, {host3.get_name()}")

stdout, stderr = host1.execute(f'python3 tools/net_llm_tester.py --test_type mtr --dest_ips {host2_ip_addr} {host3_ip_addr} --ollama_model {ollama_model} --ollama_host {ollama_fabnet_ip_addr}  --ollama_port {ollama_port}')

### Execute Traceroute Tests  

Run **traceroute** from `host1` to both `host2` and `host3`. Forward the results to the **LLM** for comparative analysis.

In [ ]:
print(f"Running traceroute test SRC: {host1.get_name()} DEST: {host2.get_name()}, {host3.get_name()}")

stdout, stderr = host1.execute(f'python3 tools/net_llm_tester.py --test_type traceroute --dest_ips {host2_ip_addr} {host3_ip_addr} --ollama_model {ollama_model} --ollama_host {ollama_fabnet_ip_addr}  --ollama_port {ollama_port}')

### Execute iPerf3 Tests  

Start **iPerf3** in **server mode** on `host2` and in **client mode** on `host1`. Send the output to the **LLM** for analysis.

In [ ]:
print(f"Starting iperf3 in server mode on {host2.get_name()}")
host2.execute_thread("export LD_LIBRARY_PATH=$LD_LIBRARY_PATH:/usr/local/lib && iperf3 -s -1")

In [ ]:
print(f"Running iperf3 test Client: {host1.get_name()} Server: {host2.get_name()}")
stdout, stderr = host1.execute(f'python3 tools/net_llm_tester.py --test_type iperf --dest_ips {host2_ip_addr} --ollama_model {ollama_model} --ollama_host {ollama_fabnet_ip_addr}  --ollama_port {ollama_port}')

### Delete the slice

In [ ]:
#slice.delete()